# LIGER best - neurips_bmmc

Seeds: `1-10`

Metrics: BAMI, CASW, SCGRAPHM, TRAJ, ISOLISI, BRAS, KBET, CPCR


In [1]:
import gc
import importlib.util
import os
import sys

os.environ["PYTHONNOUSERSITE"] = "1"
sys.path = [
    path
    for path in sys.path
    if ".local/lib/python3.10/site-packages" not in path
]

PROJECT_ROOT = '/ubc/cs/research/beaver/projects/Single-cell-integration-metrics'
os.environ["R_HOME"] = f"{PROJECT_ROOT}/CondaEnv/pyliger_env/lib/R"
sys.path.insert(0, f"{PROJECT_ROOT}/scib/")

import anndata as ad
import numpy as np
import pandas as pd

OPTIMIZER_PATH = (
    f"{PROJECT_ROOT}/scripts/optuna_multi_metrics/"
    "optuna_liger/optimize_liger_multi_metrics.py"
)
optimizer_spec = importlib.util.spec_from_file_location(
    "liger_opt",
    OPTIMIZER_PATH,
)
if optimizer_spec is None or optimizer_spec.loader is None:
    raise ImportError(f"Cannot load LIGER optimizer: {OPTIMIZER_PATH}")
liger_opt = importlib.util.module_from_spec(optimizer_spec)
sys.modules[optimizer_spec.name] = liger_opt
optimizer_spec.loader.exec_module(liger_opt)


/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATASET_NAME = 'neurips_bmmc'
DATA_PATH = f"{PROJECT_ROOT}/data/{DATASET_NAME}.h5ad"
BATCH_KEY = 'batch'
LABEL_KEY = 'cell_type'

SELECTED_BIO_METRICS = ('BAMI', 'CASW', 'SCGRAPHM', 'TRAJ', 'ISOLISI')
SELECTED_BATCH_METRICS = ('BRAS', 'KBET', 'CPCR')
SELECTED_METRICS = SELECTED_BIO_METRICS + SELECTED_BATCH_METRICS

SEEDS = tuple(range(1, 11))
MODE = "core"
LAYER = "counts"
N_NEIGHBORS = 15
EMBED_KEY = "X_liger"

liger_opt.DATASET_NAME = DATASET_NAME
liger_opt.DATA_PATH = DATA_PATH
liger_opt.BATCH_KEY = BATCH_KEY
liger_opt.LABEL_KEY = LABEL_KEY
liger_opt.MODE = MODE
liger_opt.LAYER = LAYER
liger_opt.USE_BAMI = "BAMI" in SELECTED_METRICS
liger_opt.USE_CASW = "CASW" in SELECTED_METRICS
liger_opt.USE_SCGRAPHM = "SCGRAPHM" in SELECTED_METRICS
liger_opt.USE_TRAJ = "TRAJ" in SELECTED_METRICS
liger_opt.USE_ISOLISI = "ISOLISI" in SELECTED_METRICS
liger_opt.USE_BRAS = "BRAS" in SELECTED_METRICS
liger_opt.USE_KBET = "KBET" in SELECTED_METRICS
liger_opt.USE_CPCR = "CPCR" in SELECTED_METRICS

assert DATASET_NAME == 'neurips_bmmc'
assert BATCH_KEY == 'batch'
assert LABEL_KEY == 'cell_type'
assert SELECTED_METRICS == ('BAMI', 'CASW', 'SCGRAPHM', 'TRAJ', 'ISOLISI', 'BRAS', 'KBET', 'CPCR')
assert SEEDS == tuple(range(1, 11))
assert liger_opt.CORE_MAX_ITERS == 30
assert liger_opt.N_NEIGHBORS == N_NEIGHBORS
assert liger_opt.EMBED_KEY == EMBED_KEY

pd.Series(
    {
        "dataset": DATASET_NAME,
        "data_path": DATA_PATH,
        "batch_key": BATCH_KEY,
        "label_key": LABEL_KEY,
        "metrics": list(SELECTED_METRICS),
        "seeds": list(SEEDS),
        "mode": MODE,
        "layer": LAYER,
    },
    name="configuration",
)


dataset                                           neurips_bmmc
data_path    /ubc/cs/research/beaver/projects/Single-cell-i...
batch_key                                                batch
label_key                                            cell_type
metrics      [BAMI, CASW, SCGRAPHM, TRAJ, ISOLISI, BRAS, KB...
seeds                          [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
mode                                                      core
layer                                                   counts
Name: configuration, dtype: object

In [3]:
liger_opt.validate_metric_dependencies()
liger_opt.validate_configuration(list(SELECTED_METRICS))
liger_opt.validate_dataset_contents()


Dataset preflight: n_obs=90261 n_batches=12 n_labels=45
TRAJ preflight: usable_pseudotime=51407 unique_values=51251
ISOLISI preflight: min_batch_coverage=1 isolated_labels=['CD8+ T naive CD127+ CD26- CD101-', 'T prog cycling', 'cDC1']
CPCR preflight: eligible_labels=40


In [4]:
OPTUNA_STUDY_NAME = 'liger_neurips_bmmc_joint_expanded_metrics_scgraphm_v1'
OPTUNA_BEST_TRIAL = 68
OPTUNA_BEST_VALUE = 0.7252119817841958
OPTUNA_BEST_PARAMS = {'k': 54, 'value_lambda': 6.047214048759388}

LIGER_BEST_ARGS = {
    "k": OPTUNA_BEST_PARAMS["k"],
    "value_lambda": OPTUNA_BEST_PARAMS["value_lambda"],
    "max_iters": 30,
    "mode": "core",
    "layer": "counts",
}


def run_liger_best(adata, seed):
    return liger_opt.run_liger(
        adata=adata,
        k=LIGER_BEST_ARGS["k"],
        lam=LIGER_BEST_ARGS["value_lambda"],
        max_iterations=LIGER_BEST_ARGS["max_iters"],
        mode=LIGER_BEST_ARGS["mode"],
        layer=LIGER_BEST_ARGS["layer"],
        seed=seed,
    )


pd.Series(
    {
        "study": OPTUNA_STUDY_NAME,
        "best_trial": OPTUNA_BEST_TRIAL,
        "best_value": OPTUNA_BEST_VALUE,
        **LIGER_BEST_ARGS,
    },
    name="best_setting",
)


study           liger_neurips_bmmc_joint_expanded_metrics_scgr...
best_trial                                                     68
best_value                                               0.725212
k                                                              54
value_lambda                                             6.047214
max_iters                                                      30
mode                                                         core
layer                                                      counts
Name: best_setting, dtype: object

In [5]:
def compute_joint_metrics(adata_raw, adata_int):
    return liger_opt.compute_joint_metrics(adata_raw, adata_int)


In [6]:
results = []

for seed in SEEDS:
    print(f"\n===== seed {seed} =====", flush=True)

    adata_raw = ad.read_h5ad(DATA_PATH)
    adata_int = run_liger_best(adata_raw, seed=seed)
    metric_result = compute_joint_metrics(adata_raw, adata_int)

    row = {"seed": seed, **metric_result}
    results.append(row)
    print(pd.Series(row), flush=True)

    del adata_raw, adata_int
    gc.collect()

results_df = pd.DataFrame(results)
results_df



===== seed 1 =====


/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the

Removing 228 genes not expressing in s2d5.
Removing 821 genes not expressing in s3d1.


Removing 47 genes not expressing in s4d1.
Removing 162 genes not expressing in s4d8.


Removing 73 genes not expressing in s4d9.


...

Cluster for leiden_opt_2.0 with leiden


resolution: 2.0, bAMI: 0.6518562228341294


optimised clustering against liger_metric_label
optimal cluster resolution: 0.8
optimal score: 0.6628243335687893
BRAS...


seed           10.000000
BAMI            0.662824
CASW            0.558760
SCGRAPHM        0.714855
TRAJ            0.617312
ISOLISI         0.959743
BRAS            0.819178
KBET            0.608442
CPCR            0.800613
bio_score       0.702699
batch_score     0.742744
joint_score     0.718717
dtype: float64


,seed,BAMI,CASW,SCGRAPHM,TRAJ,ISOLISI,BRAS,KBET,CPCR,bio_score,batch_score,joint_score
0,1,0.657623,0.567706,0.661199,0.663235,0.958047,0.834119,0.638588,0.809354,0.701562,0.760687,0.725212
1,2,0.658487,0.546657,0.636229,0.609527,0.951162,0.811551,0.601399,0.810502,0.680412,0.741151,0.704708
2,3,0.663610,0.553014,0.667922,0.475564,0.948178,0.814459,0.612214,0.806122,0.661657,0.744265,0.694701
3,4,0.652714,0.546666,0.673055,0.612001,0.958335,0.820788,0.616689,0.816428,0.688554,0.751301,0.713653
4,5,0.666498,0.553497,0.716216,0.501495,0.948793,0.819106,0.643573,0.812945,0.677300,0.758541,0.709796
5,6,0.659403,0.561804,0.674880,0.627589,0.968350,0.826951,0.642053,0.819490,0.698405,0.762831,0.724176
6,7,0.661233,0.549316,0.690511,0.605169,0.957411,0.813812,0.596438,0.800279,0.692728,0.736843,0.710374
7,8,0.662251,0.554333,0.634776,0.507808,0.958673,0.821893,0.636593,0.815543,0.663568,0.758010,0.701345
8,9,0.668538,0.550653,0.684625,0.603129,0.955153,0.819098,0.646490,0.816604,0.692419,0.760731,0.719744
9,10,0.662824,0.558760,0.714855,0.617312,0.959743,0.819178,0.608442,0.800613,0.702699,0.742744,0.718717


In [7]:
score_columns = [
    *SELECTED_METRICS,
    "bio_score",
    "batch_score",
    "joint_score",
]

summary_df = pd.DataFrame(
    {
        "mean": results_df[score_columns].mean(),
        "std": results_df[score_columns].std(ddof=0),
        "min": results_df[score_columns].min(),
        "max": results_df[score_columns].max(),
    }
)
summary_df


,mean,std,min,max
BAMI,0.661318,0.004321,0.652714,0.668538
CASW,0.554241,0.006427,0.546657,0.567706
SCGRAPHM,0.675427,0.026427,0.634776,0.716216
TRAJ,0.582283,0.059899,0.475564,0.663235
ISOLISI,0.956384,0.005666,0.948178,0.968350
BRAS,0.820096,0.006280,0.811551,0.834119
KBET,0.624248,0.018140,0.596438,0.646490
CPCR,0.810788,0.006376,0.800279,0.819490
bio_score,0.685931,0.014038,0.661657,0.702699
batch_score,0.751710,0.009168,0.736843,0.762831
